## Local LLMs

This notebook shows how to call a local model and have a basic chat with it.

In [11]:
import sys
import pandas as pd
import transformers, torch
from sdevpy.llms import huggingface as hfm
from sdevpy.llms import llmfactory

print(f"Python: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
print(f"Transformers: {transformers.__version__}")
print(f"PyTorch: {torch.__version__}")

# Get device information
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device type: " + device.type)

Python: 3.14.7
Transformers: 5.17.0
PyTorch: 2.14.0
Device type: mps


### Inspect available models

In [12]:
print(f"Model repo location: {hfm.model_location()}")
llm_config = llmfactory.read_llm_config()
config_df = pd.DataFrame.from_dict(llm_config, orient="index").rename_axis("model_id").fillna("")
config_df

Model repo location: /Users/sebgur/.cache/huggingface/hub


,type,repo_id,filename,name,path
model_id,,,,,
qwen3.6-35B,llama,unsloth/Qwen3.6-35B-A3B-MTP-GGUF,Qwen3.6-35B-A3B-UD-Q4_K_M.gguf,,
gemma-4-31B,llama,unsloth/gemma-4-31B-it-qat-GGUF,gemma-4-31B-it-qat-UD-Q4_K_XL.gguf,,
qwen3.6-27B,llama,unsloth/Qwen3.6-27B-GGUF,Qwen3.6-27B-Q4_K_M.gguf,,
qwen3.5-27B,llama,unsloth/Qwen3.5-27B-GGUF,Qwen3.5-27B-Q4_K_M.gguf,,
gemma-4-26B,llama,unsloth/gemma-4-26B-A4B-it-qat-GGUF,gemma-4-26B-A4B-it-qat-UD-Q4_K_XL.gguf,,
gemma-4-12B,llama,bartowski/gemma-4-12B-it-GGUF,gemma-4-12B-it-Q4_K_M.gguf,,
qwen3.5-9B,llama,unsloth/Qwen3.5-9B-GGUF,Qwen3.5-9B-Q4_K_M.gguf,,
qwen3.5-4B,llama,unsloth/Qwen3.5-4B-GGUF,Qwen3.5-4B-Q4_K_M.gguf,,
gemma-4-E4B,llama,unsloth/gemma-4-E4B-it-qat-GGUF,gemma-4-E4B-it-qat-UD-Q4_K_XL.gguf,,


### Chat with the model
Pick a model and then test it with a little chat.

In [13]:
# Choose the model
model_id = "gemma-4-12B"
# model_id = "qwen3.6-35B"

# Load the model into RAM
model = llmfactory.from_pretrained(model_id, max_context_tokens=8192)

# Display characteristics
model.pretty_print()

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 2048
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 2048


Model path: /Users/sebgur/.cache/huggingface/hub/models--bartowski--gemma-4-12B-it-GGUF/snapshots/2ae7d41be21ca62de00a2d320ee9cec50daa3aa6/./gemma-4-12B-it-Q4_K_M.gguf
Model name: Gemma 4 12B It
Architecture: gemma4
Context length: 8192
File size (GB): 7.136


In [14]:
# Respond to simple prompt
prompt = "Why is the sky blue?"
response = model.respond_prompt(prompt, max_tokens=1000)
print(response)

The sky is blue because of a phenomenon called **Rayleigh scattering**.

Here is the step-by-step breakdown of how it works:

### 1. Sunlight is a mix of all colors
Although sunlight looks white, it is actually made up of all the colors of the rainbow (red, orange, yellow, green, blue, indigo, and violet). Each color travels as a wave with a different wavelength. Red light has long, lazy waves; blue and violet light have much shorter, choppier waves.

### 2. The Atmosphere is an obstacle course
Earth's atmosphere is filled with gases (mostly nitrogen and oxygen) and tiny particles. When sunlight hits these gas molecules, the light bumps into them and scatters in different directions.

### 3. Short waves scatter more
Because blue light travels in shorter, smaller waves, it strikes the gas molecules and gets scattered much more easily than red light. Imagine throwing large boulders (red light) and small marbles (blue light) through a forest; the marbles will hit branches and bounce every

### Execute an instruction
Here we are going to ask the model to read an email and answer some questions.

In [6]:
# Email
email = """ Subject: Work harder, you lazy employee

Dear Alexandra,

I hope this message finds you well. I am enquiring about the new project "do something useless and be proud", which,
as far as I remember is due for completion next month. Would you be able to let me know what is the exact date for
delivery?
Furthermore, we would like to add to the project a new task where the system will tell us everything we want to know
very fast without paying any additional cost. Would it be possible to 'squeeze that in' the project, without, of course
changing any date for delivery?

It is very urgent for us to receive it as the Very Import People Committee considers it crucial for running its
business.

We are aware that you are currently working on what we previously asked you as high priority, but as this is so
important, we would appreciate it if you could expedite the work using any means necessary. Since we think it is
such an important task and we know better, we don't expect any issue with this request, but will not hesitate to
escalate to Management if it were to be questioned.

Many thanks,

John Doe,
Very Important People Committee,
Motivation&Spirit Ltd.
Tel: +1 222 333 4444
"""

In [7]:
# Set up the prompt
system_prompt = """ You are a first line responder to internal client requests. You know that your internal systems have
                    been built by piling up workarounds on workarounds because the said internal clients behave like
                    giant babies who never want to wait for anything. You work under constant pressure to achieve
                    completely impossible tasks and burn out is never really far away. """

user_prompt = f""" Write a response email, which remains polite but expresses the fact that it will not be possible to
                   meet unreasonable requests for additional work. Cynicism and sarcasm are encouraged in the response.
                   Here is the email to respond to: {email} """

In [8]:
# Set response parameters
temperature, top_p, num_beams, max_tokens = 0.4, 0.8, 1, 4096

# Build message input
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]

# Query
response = model.chat(messages, temperature=temperature, top_p=top_p, num_beams=num_beams,
                      max_tokens=max_tokens)
print(response)

Subject: Re: Work harder, you lazy employee / Project "do something useless and be proud"

Dear John,

Thank you for your email and for taking the time to outline your expectations. I truly appreciate the committee’s unwavering confidence in my ability to bend the laws of software architecture, basic project management, and the conservation of time to your whims.

Regarding the delivery date for “do something useless and be proud,” it remains scheduled for next month, as originally agreed. I’ll be sure to notify you the moment the calendar spontaneously rearranges itself to accommodate earlier deadlines.

As for the additional requirement—having the system instantly provide all desired insights without any additional cost or timeline adjustment—I’ve forwarded this to our engineering team. They’re currently drafting a patch that will also allow me to work 48 hours a day, maintain perfect sleep hygiene, and compile legacy code written by a team that vanished in 2012. Unfortunately, the p

In [15]:
# Unload the model from RAM
model.unload()